# Lesson 1a: From Linear Models to Neurons — Theory

In the previous notebook we proved that a *single* linear model has a hard
ceiling — it cannot separate XOR, no matter how its weights are chosen — and
we hand-built a two-layer network that clears that ceiling by combining two
linear boundaries through a non-linearity.

But we hand-derived those weights. Nobody hands you the weights in practice.
This notebook answers the next question: **how does a single unit learn its
own weights from data?** We will show that logistic regression — a
classifier you likely already know — *is* a single artificial neuron, derive
its loss and gradient from first principles, and then implement three
flavors of gradient descent (batch, mini-batch, stochastic) entirely in
NumPy to watch that single neuron learn.

By the end of this notebook you will have:
- derived the classical **perceptron** update rule and seen why its loss is
  not smooth,
- derived **logistic regression** as a single neuron (weighted sum +
  sigmoid), stated its cross-entropy loss, and derived the gradient step by
  step,
- implemented **batch, mini-batch, and stochastic gradient descent** from
  scratch and compared their convergence, and
- shown empirically how the **learning rate** governs convergence, including
  a case where it makes training diverge.

## Introduction

Recall the factory quality-control problem from Lesson 0a, but now
imagine a *linearly separable* variant: a sensor line where two readings,
temperature and vibration, reliably predict "pass" or "fail" with a single
straight cut. You don't need a whole hidden layer for this — a single linear
decision boundary suffices.

This is exactly the setting for a **single neuron**. A neuron computes a
weighted sum of its inputs, adds a bias, and passes the result through an
activation function. Two classical machine learning algorithms are special
cases of this one computation:

- the **perceptron** (Rosenblatt, 1958) uses a hard step activation and a
  mistake-driven update rule,
- **logistic regression** uses a smooth sigmoid activation and is trained by
  gradient descent on a probabilistic loss.

Both are single neurons. The perceptron is the ancestor; logistic regression
is the version whose smooth loss makes gradient-based learning well-behaved,
and it is *exactly* the computation performed by one output unit of a
modern neural network. Understanding it deeply is what makes the jump to
multi-layer networks (Lesson 2a) a small step rather than a leap.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed seed: every random draw in this notebook is reproducible.
SEED = 0
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)

### A small, linearly-separable dataset

To keep every run fast (this notebook must execute in well under ten
minutes on a CPU) and every plot easy to read, we use a synthetic two-blob
dataset rather than raw MNIST pixels: two Gaussian clusters in 2D, one
labeled 0 and one labeled 1, with a little overlap so the problem is not
trivially separable — reminiscent of an MNIST "0 vs. 1" pixel-intensity
classification task, but visualizable directly. Every idea developed here
(perceptron update, logistic loss, batch/mini-batch/SGD, learning rate)
applies unchanged to a real pixel dataset like MNIST digits.

In [ ]:
N_PER_CLASS = 150

mean0, mean1 = np.array([-1.5, -1.0]), np.array([1.5, 1.0])
cov = np.array([[1.0, 0.3], [0.3, 1.0]])

X0 = rng.multivariate_normal(mean0, cov, size=N_PER_CLASS)
X1 = rng.multivariate_normal(mean1, cov, size=N_PER_CLASS)

X = np.vstack([X0, X1])
y = np.concatenate([np.zeros(N_PER_CLASS), np.ones(N_PER_CLASS)])

# Shuffle once, up front, so batch order is not correlated with class.
perm = rng.permutation(len(X))
X, y = X[perm], y[perm]

print("X shape:", X.shape, " y shape:", y.shape)
print("class balance:", y.mean())

colors = np.where(y == 1, "crimson", "steelblue")
plt.scatter(X[:, 0], X[:, 1], c=colors, s=30, edgecolor="k", linewidth=0.3, alpha=0.8)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("Synthetic two-blob dataset (crimson = 1, steelblue = 0)")
plt.grid(alpha=0.3)
plt.show()

## The Perceptron

The perceptron is the oldest trainable linear classifier. It computes

$$
z = w^\top x + b, \qquad \hat{y} = \text{step}(z) = \begin{cases} 1 & z \geq 0 \\ 0 & z < 0 \end{cases}
$$

and learns with the **perceptron update rule**: for each training example
$(x_i, y_i)$, if the prediction is wrong, nudge the weights toward the
correct answer —

$$
w \leftarrow w + \eta (y_i - \hat{y}_i) x_i, \qquad b \leftarrow b + \eta (y_i - \hat{y}_i)
$$

and if $\hat{y}_i = y_i$, do nothing. This is a *mistake-driven* rule: it
only fires on misclassified points, and it is guaranteed to converge to a
zero-error solution in finitely many steps **if and only if the data is
linearly separable** (the Perceptron Convergence Theorem). If the classes
overlap even slightly — as ours do — the perceptron never settles; the
weights keep oscillating as it keeps trying to fix examples it structurally
cannot.

The step function is the crux of both the perceptron's strength and its
weakness. It gives clean, binary decisions, but it is **flat almost
everywhere** ($\frac{d}{dz}\,\text{step}(z) = 0$ everywhere except an
undefined jump at $z=0$). There is no useful gradient to follow — you cannot
ask "how much would nudging $w$ by $\epsilon$ improve the loss?" because the
step function's slope tells you nothing. That is precisely why the
perceptron rule is mistake-driven rather than gradient-based, and precisely
what logistic regression fixes by swapping the step for a smooth sigmoid.

In [ ]:
def step(z):
    return (z >= 0).astype(float)

def train_perceptron(X, y, lr=0.1, epochs=40):
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes_per_epoch = []
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            z = w @ xi + b
            yhat = step(np.array([z]))[0]
            error = yi - yhat
            if error != 0:
                w += lr * error * xi
                b += lr * error
                mistakes += 1
        mistakes_per_epoch.append(mistakes)
    return w, b, mistakes_per_epoch

w_p, b_p, mistakes = train_perceptron(X, y, lr=0.1, epochs=40)
print(f"final w={w_p}, b={b_p:.3f}")
print(f"mistakes in final epoch: {mistakes[-1]} / {len(X)}")

In [ ]:
plt.plot(mistakes, marker="o", markersize=3)
plt.xlabel("epoch")
plt.ylabel("# misclassified points")
plt.title("Perceptron: mistakes per epoch (overlapping data never reaches 0)")
plt.grid(alpha=0.3)
plt.show()

On our overlapping blobs the mistake count does not reach zero and
keeps bouncing — exactly what the convergence theorem predicts for
non-separable data. The perceptron gives us no notion of "how wrong" a
prediction was (only right/wrong), and no smooth loss surface to descend.
Logistic regression fixes both problems at once.

## Logistic Regression as a Neuron

Replace the perceptron's hard step with the **sigmoid** activation:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = w^\top x + b, \qquad \hat{y} = \sigma(z) \in (0, 1)
$$

This single computation — a weighted sum followed by a non-linear activation
— is exactly what one artificial neuron computes. Logistic regression *is*
a single neuron; nothing more needs to be added to get from "linear model"
to "neuron" except this one non-linearity. Because $\sigma(z) \in (0, 1)$,
we interpret $\hat{y}$ as $P(y=1 \mid x)$.

### Binary cross-entropy loss

We want the model's predicted probability to match the true label. The
natural loss for this is the negative log-likelihood of the true label under
the model, better known as **binary cross-entropy**:

$$
\mathcal{L}(w, b) = -\frac{1}{n}\sum_{i=1}^n \Big[ y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i) \Big], \qquad \hat{y}_i = \sigma(w^\top x_i + b)
$$

For a single example, if $y_i = 1$ the loss is $-\log \hat{y}_i$ (heavily
penalizes confident wrong answers, $\hat y_i \to 0$, and vanishes as
$\hat y_i \to 1$); if $y_i = 0$ the loss is $-\log(1-\hat{y}_i)$, symmetric
the other way. Unlike the perceptron's flat 0/1 mistake count, this loss is
smooth and differentiable everywhere in $w$ and $b$ — which is exactly what
lets us follow a gradient downhill.

### Deriving the gradient, step by step

We need $\frac{\partial \mathcal{L}}{\partial w}$ for one example
$(x, y)$ and its prediction $\hat y = \sigma(z)$, $z = w^\top x + b$, then
average over the dataset.

**Step 1 — derivative of the sigmoid.** A useful identity:
$$
\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)
$$
(Differentiate $\sigma(z) = (1+e^{-z})^{-1}$ directly with the chain rule to
confirm this.)

**Step 2 — loss for one example**, writing $\ell = -\big[y\log\hat y + (1-y)\log(1-\hat y)\big]$:
$$
\frac{\partial \ell}{\partial \hat y} = -\frac{y}{\hat y} + \frac{1-y}{1-\hat y}
= \frac{\hat y - y}{\hat y (1 - \hat y)}
$$

**Step 3 — chain rule through the sigmoid**, using $\frac{\partial \hat y}{\partial z} = \sigma'(z) = \hat y(1-\hat y)$:
$$
\frac{\partial \ell}{\partial z} = \frac{\partial \ell}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}
= \frac{\hat y - y}{\hat y(1-\hat y)} \cdot \hat y (1 - \hat y) = \hat y - y
$$

This cancellation is the entire reason cross-entropy and sigmoid are paired
in practice: the messy $\hat y(1-\hat y)$ denominator from the loss exactly
cancels the same factor from the sigmoid's derivative, leaving the simplest
possible error signal, $(\hat y - y)$.

**Step 4 — chain rule through the linear layer**, using $\frac{\partial z}{\partial w} = x$ and $\frac{\partial z}{\partial b} = 1$:
$$
\frac{\partial \ell}{\partial w} = (\hat y - y)\, x, \qquad \frac{\partial \ell}{\partial b} = \hat y - y
$$

**Step 5 — average over the dataset** (batch gradient, $n$ examples):
$$
\nabla_w \mathcal{L} = \frac{1}{n}\sum_{i=1}^n (\hat y_i - y_i)\, x_i, \qquad
\nabla_b \mathcal{L} = \frac{1}{n}\sum_{i=1}^n (\hat y_i - y_i)
$$

The final gradient descent update, with learning rate $\eta$, is:
$$
w \leftarrow w - \eta\, \nabla_w \mathcal{L}, \qquad b \leftarrow b - \eta\, \nabla_b \mathcal{L}
$$

Compare this to the perceptron update: both move $w$ in the direction of
$x$ scaled by an error term ($y-\hat y$ there, $\hat y - y$ here, sign
flipped only by which side subtracts which). The crucial difference is that
the perceptron's error is binary (a discrete right/wrong from the step
function) while logistic regression's error $(\hat y - y)$ is continuous and
proportional to *how wrong* the probability estimate is — which is what
gives gradient descent something informative to follow at every point, not
just at misclassifications.

In [ ]:
def sigmoid(z):
    # Numerically stable sigmoid (clip to avoid overflow in exp).
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def bce_loss(w, b, X, y, eps=1e-12):
    z = X @ w + b
    yhat = sigmoid(z)
    yhat = np.clip(yhat, eps, 1 - eps)
    return -np.mean(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))

def bce_gradient(w, b, X, y):
    n = X.shape[0]
    z = X @ w + b
    yhat = sigmoid(z)
    error = yhat - y                 # shape (n,)
    grad_w = X.T @ error / n         # shape (d,)
    grad_b = np.mean(error)
    return grad_w, grad_b

# Sanity check: numerical gradient vs analytic gradient at a random point.
w0 = rng.normal(size=X.shape[1]) * 0.1
b0 = 0.05
grad_w_analytic, grad_b_analytic = bce_gradient(w0, b0, X, y)

eps = 1e-5
grad_w_numeric = np.zeros_like(w0)
for j in range(len(w0)):
    w_plus, w_minus = w0.copy(), w0.copy()
    w_plus[j] += eps; w_minus[j] -= eps
    grad_w_numeric[j] = (bce_loss(w_plus, b0, X, y) - bce_loss(w_minus, b0, X, y)) / (2 * eps)

grad_b_numeric = (bce_loss(w0, b0 + eps, X, y) - bce_loss(w0, b0 - eps, X, y)) / (2 * eps)

print("analytic grad_w:", grad_w_analytic, " numeric grad_w:", grad_w_numeric)
print(f"analytic grad_b: {grad_b_analytic:.6f}  numeric grad_b: {grad_b_numeric:.6f}")
max_diff = max(np.abs(grad_w_analytic - grad_w_numeric).max(), abs(grad_b_analytic - grad_b_numeric))
print(f"max abs difference: {max_diff:.2e}")
assert max_diff < 1e-4, "analytic and numerical gradients should agree closely"
print("Analytic gradient matches finite-difference gradient.")

## Gradient Descent Variants

We now train the same logistic-regression objective three different
ways, all using the exact gradient derived above, differing only in *how
much data* each update sees:

- **Batch gradient descent** — every update uses the *entire* dataset (the
  formula above, exactly).
- **Mini-batch gradient descent** — every update uses a small random subset
  (say 32 examples), re-shuffled each epoch.
- **Stochastic gradient descent (SGD)** — every update uses a *single*
  randomly chosen example ($n=1$ in the formula above).

All three descend the same loss surface; they differ in how noisy each
step's gradient estimate is. We track the *full-dataset* loss after every
individual update so all three curves are on a directly comparable x-axis
(number of gradient steps taken).

In [ ]:
def init_params(d, seed):
    g = np.random.default_rng(seed)
    return g.normal(scale=0.1, size=d), 0.0

def gd_batch(X, y, lr=0.5, steps=200, seed=SEED):
    w, b = init_params(X.shape[1], seed)
    losses = []
    for _ in range(steps):
        grad_w, grad_b = bce_gradient(w, b, X, y)
        w -= lr * grad_w
        b -= lr * grad_b
        losses.append(bce_loss(w, b, X, y))
    return w, b, losses

def gd_minibatch(X, y, lr=0.5, steps=200, batch_size=32, seed=SEED):
    w, b = init_params(X.shape[1], seed)
    g = np.random.default_rng(seed + 1)
    n = X.shape[0]
    losses = []
    idx = g.permutation(n)
    pos = 0
    for _ in range(steps):
        if pos + batch_size > n:
            idx = g.permutation(n)
            pos = 0
        batch = idx[pos:pos + batch_size]
        pos += batch_size
        grad_w, grad_b = bce_gradient(w, b, X[batch], y[batch])
        w -= lr * grad_w
        b -= lr * grad_b
        losses.append(bce_loss(w, b, X, y))
    return w, b, losses

def gd_sgd(X, y, lr=0.5, steps=200, seed=SEED):
    w, b = init_params(X.shape[1], seed)
    g = np.random.default_rng(seed + 2)
    n = X.shape[0]
    losses = []
    idx = g.permutation(n)
    pos = 0
    for _ in range(steps):
        if pos >= n:
            idx = g.permutation(n)
            pos = 0
        i = idx[pos]
        pos += 1
        grad_w, grad_b = bce_gradient(w, b, X[i:i+1], y[i:i+1])
        w -= lr * grad_w
        b -= lr * grad_b
        losses.append(bce_loss(w, b, X, y))
    return w, b, losses

STEPS = 200
LR = 0.5
w_batch, b_batch, loss_batch = gd_batch(X, y, lr=LR, steps=STEPS)
w_mini, b_mini, loss_mini = gd_minibatch(X, y, lr=LR, steps=STEPS, batch_size=32)
w_sgd, b_sgd, loss_sgd = gd_sgd(X, y, lr=LR, steps=STEPS)

print(f"final full-dataset loss — batch: {loss_batch[-1]:.4f}, "
      f"mini-batch: {loss_mini[-1]:.4f}, sgd: {loss_sgd[-1]:.4f}")

In [ ]:
plt.plot(loss_batch, label="batch (n=300/step)")
plt.plot(loss_mini, label="mini-batch (n=32/step)")
plt.plot(loss_sgd, label="SGD (n=1/step)")
plt.xlabel("gradient step")
plt.ylabel("full-dataset BCE loss")
plt.title("Convergence: batch vs. mini-batch vs. stochastic GD")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Batch gradient descent uses the true full-dataset gradient at every
step, so its loss curve is smooth and monotone. Mini-batch and SGD use noisy
*estimates* of that gradient — SGD the noisiest of all, since a single
example is a very rough proxy for the dataset's true gradient — so their
curves are visibly jagged even as they trend downward at a similar or
faster *wall-clock* rate (each of their "steps" is far cheaper to compute).
This bias/variance trade-off — noisier but cheaper updates — is the
practical reason mini-batch SGD, not full-batch descent, is the default for
training real neural networks on large datasets.

## Learning Rate Effects

The learning rate $\eta$ scales every gradient step. Too small, and
training crawls; too large, and the update can overshoot the minimum so
badly that the loss grows step over step instead of shrinking — the
classic **divergence** failure mode. We compare four learning rates on the
same batch gradient descent setup to make the effect visible on one loss
surface.

In [ ]:
learning_rates = [0.01, 0.5, 3.0, 2000.0]
lr_losses = {}
for lr in learning_rates:
    _, _, losses = gd_batch(X, y, lr=lr, steps=100)
    lr_losses[lr] = losses

for lr, losses in lr_losses.items():
    tag = "DIVERGES/OSCILLATES" if max(losses) > 2 * losses[0] else "converges"
    print(f"lr={lr:>7}: loss[0]={losses[0]:.3f}  loss[-1]={losses[-1]:.3f}  "
          f"max={max(losses):.3f}  ({tag})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for lr, losses in lr_losses.items():
    axes[0].plot(losses, label=f"lr={lr}")
axes[0].set_xlabel("gradient step"); axes[0].set_ylabel("BCE loss")
axes[0].set_title("Linear scale (divergent run dwarfs the rest)")
axes[0].legend(); axes[0].grid(alpha=0.3)

for lr, losses in lr_losses.items():
    axes[1].plot(np.log10(np.array(losses) + 1e-12), label=f"lr={lr}")
axes[1].set_xlabel("gradient step"); axes[1].set_ylabel("log10(BCE loss)")
axes[1].set_title("Log scale (same data, all curves visible)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

assert max(lr_losses[2000.0]) > 3 * lr_losses[2000.0][0], \
    "lr=2000 should blow up well past its starting loss at some point during training"
assert lr_losses[0.5][-1] < lr_losses[0.5][0] * 0.5, "lr=0.5 should converge well"
print("Confirmed: lr=2000 blows up/oscillates far above its starting loss; "
      "lr=0.5 converges comfortably.")

- **lr = 0.01** converges, but slowly — after 100 steps it has barely
  moved compared to the larger, well-tuned rates.
- **lr = 0.5** converges quickly and smoothly to a low loss — a
  well-chosen rate for this problem.
- **lr = 3.0** overshoots the minimum on every step, producing a visibly
  oscillatory but still (on average) decreasing loss.
- **lr = 2000.0** diverges outright: each step overshoots so far past the
  minimum that the next gradient is even larger, and the loss repeatedly
  spikes to many times its starting value instead of settling — visible on
  the linear-scale plot as sharp explosions and on the log-scale plot as a
  jagged line that never comes down to the other rates' level.

This is why learning rate is the single most consequential hyperparameter
in gradient-based learning, and why real training pipelines invest in
learning-rate schedules, warmup, and adaptive optimizers (Adam and friends)
— all of which exist to keep the step size in the narrow band between "too
slow to be useful" and "large enough to diverge."\n

## Key Takeaways

- The **perceptron** and **logistic regression** are both single
  neurons — a weighted sum plus an activation — differing only in that the
  perceptron uses a hard step (mistake-driven, no useful gradient, only
  guaranteed to converge on linearly separable data) while logistic
  regression uses a smooth sigmoid (gradient-based, well-defined on any
  data).
- Logistic regression's **binary cross-entropy loss**, paired with the
  **sigmoid** activation, produces a remarkably simple gradient:
  $\nabla_w \mathcal{L} = \frac{1}{n}\sum_i (\hat y_i - y_i) x_i$ — the
  messy sigmoid-derivative and cross-entropy-derivative terms cancel
  exactly, leaving "predicted probability minus true label" as the entire
  error signal. We verified this analytic gradient against a numerical
  finite-difference gradient.
- **Batch, mini-batch, and stochastic gradient descent** all descend the
  same loss surface with the same gradient formula, differing only in how
  much data each step's gradient estimate uses. Batch is smooth but
  expensive per step; SGD is noisy but cheap per step; mini-batch is the
  practical middle ground used almost universally in real training.
- The **learning rate** controls the step size along the gradient. Too
  small wastes compute; too large causes oscillation; too large still
  causes outright **divergence** — the loss grows instead of shrinking. We
  showed all three regimes on the same objective.
- Everything derived here — the weighted-sum-plus-activation computation,
  the loss, the gradient, and the three gradient descent variants — carries
  over unchanged when we stack many neurons into layers (Lesson 2a) and
  reproduce this exact model in PyTorch (Lesson 1b).